In [1]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_cohere import ChatCohere


load_dotenv()

True

In [2]:
import asyncio
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult,TextContent
from langchain_mcp_adapters.client import MultiServerMCPClient

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Interpret MCP tool calls: retry transient failures, surface all errors gracefully.

    -Retryable Mcp Error codes (e.g. -32603): retry with exponential backoff.
    -Non-Retryable Mcp error codes (e.g. -32602): return error message immediately.
    -Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on request.name"
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}) : {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(content=[TextContent(type="text", text=f"Tool Call failed (non-retryable) : {exc}")],
                                          isError=False,
                                          )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name}"
                      f"(attempt {attempt+1}/{self.max_retries}) : {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2**attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )


client = MultiServerMCPClient(
    {
        "travel_server" : {
            "transport" : "http",
            "url" : "https://mcp.kiwi.com"
        }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [4]:
from typing import Dict,Any
from tavily import TavilyClient
from langchain.tools import tool


tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message" : "Search limit reached please summarize your findings and provide your final answer."}

    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error" : str(e)}

In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database:  {e}"

In [3]:
model = ChatCohere(model="command-r-08-2024", temperature=0)
travel_agent = create_agent(model,
                     tools=tools,
                     checkpointer=InMemorySaver(),
                     system_prompt="You are a travel agent. Your job is to know the users needs and recommend flights according to that and No Follow UP Questions"
                     )

In [5]:

response = await travel_agent.ainvoke(
    {"messages" : [HumanMessage(content="Book me a flight from Patna to Delhi one passenger for 5 October")]},
    {"configurable" : {"thread_id" : "flight"}}
)

print(response["messages"][-1].content)


I've found a number of flights from Patna to Delhi on 26 September 2026. Here are the cheapest and shortest options:

## Cheapest
- Route: Patna → Delhi
- Times & duration: 22:10 → 23:45 (1h 35m)
- Cabin: Economy
- Price: 70 EUR
- Booking link: https://kiwi.com/u/wjr97q

## Shortest
- Route: Patna → Delhi
- Times & duration: 10:20 → 12:20 (2h)
- Cabin: Economy
- Price: 87 EUR
- Booking link: https://kiwi.com/u/b3wqcw

I recommend the shortest option, which is also the most expensive. Have a nice trip!


In [5]:
model = ChatCohere(model="command-r-08-2024", temperature=1.0)
venue_agent = create_agent(model,
                           tools=[web_search],
                           checkpointer=InMemorySaver(),
                           system_prompt="You are a venue Planner for the Wedding, Your Job is to Recommend the best Venue according to User's need")

In [12]:
response =  venue_agent.invoke({"messages" : [HumanMessage(content="I want to have the wedding in a place which is near sea")]},
                                    {"configurable" : {"thread_id" : "venue"}})

print(response["messages"][-1].content)

Certainly! Having a wedding near the sea is a wonderful idea. Here are some additional venue options to consider:

1. Oceanfront Hotel: Look for a hotel or resort that boasts an oceanfront location. These venues often have dedicated wedding coordinators and can offer a range of packages to suit your needs. You can have your ceremony on the beach, followed by a reception in a beautiful ballroom or outdoor terrace with breathtaking sea views.

2. Lighthouse Wedding: If you're seeking a unique and iconic setting, consider a lighthouse wedding. Many lighthouses offer event spaces with panoramic views of the ocean. The historic charm and dramatic backdrop of a lighthouse can create an unforgettable wedding experience.

3. Private Beach Club: Exclusive beach clubs often provide a luxurious and intimate setting for weddings. These venues typically have private beaches, elegant dining areas, and stunning sea vistas. A beach club wedding ensures a relaxed and sophisticated atmosphere for you an

In [6]:
model = ChatCohere(model="command-r-08-2024", temperature=1.0)
music_agent = create_agent(model,
                           tools=[web_search],
                           checkpointer=InMemorySaver(),
                           system_prompt="""You are a Wedding Music Curator Your job is to match music genres and styles according to what the couple and their guests like, Ask about their Favorite Artists, the vibe they want(e.g. Romantic, energetic, beachy) and Recommend Specific Genres""")

In [7]:
response = music_agent.invoke({"messages" : [HumanMessage(content="Well Romantic songs now you suggest the artists and songs")]},
                              {"configurable" : {"thread_id" : "music"}})

print(response["messages"][-1].content)


KeyboardInterrupt



In [8]:
@tool
def music_specialist(query:str)-> str:
    """Delegate Music genres, Artists and specific Music to the Lead Agent"""

    result = music_agent.invoke({"messages" : [HumanMessage(content=query)]},
                              {"configurable" : {"thread_id" : "music"}})

    return result["messages"][-1].content

In [9]:
@tool
def venue_specialist(query:str) -> str:
    """Delegate Venue Place, Pricing and Specifications of Venue to the Lead Agent"""

    result = venue_agent.invoke({
        "messages" : [HumanMessage(content=query)]
    },
        {"configurable" : {"thread_id" : "venue"}})

    return result["messages"][-1].content

In [10]:
@tool
async def travel_specialist(query: str) -> str:
    """Delegate flight, pricing, and travel search requests to the Lead Agent"""

    result = await travel_agent.ainvoke(
        {"messages" : [HumanMessage(content=query)]},
        {"configurable" : {"thread_id" : "isolated_travel_thread"}}
    )

    return result["messages"][-1].content


In [11]:
model = ChatCohere(model="command-r-08-2024", temperature=0)
agent = create_agent(model,
                     tools = [venue_specialist, travel_specialist, music_specialist],
                     checkpointer=InMemorySaver(),
                     system_prompt="""You are the Lead Wedding Planner, Your job is to Orchestrate the User's Wedding , By Delegating tasks to your specialized Department (Travel, Venue and Music). Analyze the User's Request, use the appropriate rule to get the necessary information and synthesize a final response.""")

In [12]:
while True:
    user_input = input("Put your query Here or type quit to exit")
    if user_input == "quit":
        break
    response = await agent.ainvoke({"messages" : [HumanMessage(content=user_input)]},
                                   {"configurable" : {"thread_id" : "agent"}})
    print(response["messages"][-1].content)

Great! I'm happy to help you plan your wedding.

## Venue
Can you tell me more about your vision for the wedding and the kind of venue you're looking for?

## Travel
What kind of travel help do you need?

## Music
I've suggested some popular romantic artists and songs that might appeal to a broad range of tastes. Would you like me to suggest some more romantic artists and songs, or perhaps recommend some genres based on your preferences?

- Ed Sheeran - "Perfect"
- Adele - "Someone Like You"
- John Legend - "All of Me"
- Whitney Houston - "I Will Always Love You"
- Mariah Carey - "Vision of Love"
- Bruno Mars - "Just the Way You Are"
- Luis Fonsi & Daddy Yankee - "Despacito"
- Justin Timberlake - "Can't Stop the Feeling!"
- Michael Bublé - "Everything"
- Ella Mai - "Boo'd Up"
I'm sorry, I was unable to find any flights for the dates you provided. Please provide dates that are today or later.

Here are some seaside venues for a wedding:

- Auberge resort Esperanza, Cabo
- Rockhouse hote